# MiniMax H3 Studio（ジョブは1つ。生成は人間）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fireworker011/Research/blob/cursor/h3-sfw-colab-6dc5/minimax_h3_studio_bot.ipynb)

コードセルはプランとプロンプトだけを書く。Comfy は起動しない。公式 Hailuo API にも投げない。
Drive の `minimax-h3-comfyui/studio/latest-plan.json` に置く。`episodes/` には書かない。

Look の空欄はプロンプトに出ない。swap は Hero シート（絵のパス）が要る。
action は英語1本。日本語は「」か DIALOGUE。商品名は書かない。
runtime が api で Combat / Swap / real を選ぶと止まる。

- `fast_motion` — FL2VA。スピード LoRA 1.0 は常に載る。アクション 0.6 とコンバット 0.7 は任意。格闘トリガーは付けない。
- `combat_motion` — FL2VA + combat 1.0。Turbo 切。High-Mem。finish で決め。スピードやアクションとは積まない。
- `swap_character` — キャラ差し替え。Ref2VA + charswap 1.0。Video=動き、Picture=全身。Hero シート必須。
- `swap_face` — 同じ charswap。Picture は顔と髪だけ。服は Video。
- `swap_outfit` — 同じ charswap。Picture は服だけ。顔は Video。
- `real` — anime2real 1.0。swap と同じサンプラーには積まない。
- `orbit360` — FL2VA。同じ絵を首尾。LoRA ファイル名は未確定なので積まない。
- `two_pass` — A の mp4 を B の Video 1。LoRA はパスごとに分ける。
- `text_scene` — シーン。T2VA。LoRA なし。場所は Hero の place。空欄は書かない。
- `join` — 6秒と9秒を切って足す。15秒以上。1本の15秒生成はしない。
- `affi_template` — バズ型の既存台本を指す。台本は書き換えない。生成しない。

`fast_motion` はスピード LoRA（Turbo 8step、steps 9、video shift 6）。アクション 0.6 とコンバット 0.7 はチェックで足す。格闘の `prfight2` は `combat_motion` だけ。
キャラは Hero の項目と `swap_character`。場所は Hero / Enemy の place。空欄は Look に出ない。シーンは `text_scene`。
`join` は 6 秒と 9 秒を足して 15 秒以上にする。1 本の 15 秒生成はしない。ffmpeg は実行しない。
`affi_template` は既存のバズ型台本を指す。台本は書き換えない。
Combat の格闘と Swap は Turbo を切る。本体は Comfy High-Mem。
Weapon / GunFu / Continuity は無い。orbit の LoRA ファイル名は未確定。アダルト用ノートはこのブランチに無い。
生成ノートは `h3-runner/minimax_h3_still.ipynb`。

下の2つ目のコードセルは `text_scene` を7本書く。公式 Hailuo の T2VA。LoRA なし。Combat も charswap も切る。各本6秒、16:9、768P、24fps、1ショット。ポスターは静止画。顔固定はこの7本に載せない。生成ボタンは押さない。
並び: `hana-gate`、`host-live`、`hana-cart`、`hana-shelf`、`host-drop`、`hana-box`、`hana-exit`。
セッション名 `h3-studio`。


In [ ]:
#@title H3 Studio：ジョブを1つ選ぶ（生成はしない）
#@markdown 病棟エンジンは使わない。プランとプロンプトだけ書く。Comfy High-Mem の生成ボタンは人間が押す。
JOB = "text_scene"  #@param ["fast_motion", "combat_motion", "swap_character", "swap_face", "swap_outfit", "real", "orbit360", "two_pass", "text_scene", "join", "affi_template"]
RUNTIME = "comfy"  #@param ["comfy", "api"]
HIGH_MEM = True  #@param {type:"boolean"}
FINISH = False  #@param {type:"boolean"}
TURBO = False  #@param {type:"boolean"}
DURATION = "5"  #@param ["5", "4"]
FAST_SECONDS = "6"  #@param ["6", "9"]
WITH_ACTION = False  #@param {type:"boolean"}
WITH_COMBAT = False  #@param {type:"boolean"}
TEMPLATE = "buy_before"  #@param ["buy_before", "daily_food", "daily_camera"]
PARTS = "6,9"  #@param {type:"string"}
ACTION = ""  #@param {type:"string"}
DIALOGUE = ""  #@param {type:"string"}
#@markdown **Hero。空欄は Look に書かない。**
HERO_HAIR = ""  #@param {type:"string"}
HERO_COLOR = ""  #@param {type:"string"}
HERO_RACE = ""  #@param {type:"string"}
HERO_AGE = ""  #@param {type:"string"}
HERO_HEIGHT = ""  #@param {type:"string"}
HERO_WEIGHT = ""  #@param {type:"string"}
HERO_CLOTHES = ""  #@param {type:"string"}
HERO_PLACE = ""  #@param {type:"string"}
#@markdown **Enemy。空欄は Look に書かない。**
ENEMY_HAIR = ""  #@param {type:"string"}
ENEMY_COLOR = ""  #@param {type:"string"}
ENEMY_RACE = ""  #@param {type:"string"}
ENEMY_AGE = ""  #@param {type:"string"}
ENEMY_HEIGHT = ""  #@param {type:"string"}
ENEMY_WEIGHT = ""  #@param {type:"string"}
ENEMY_CLOTHES = ""  #@param {type:"string"}
ENEMY_PLACE = ""  #@param {type:"string"}
HERO_SHEET = ""  #@param {type:"string"}
VIDEO = ""  #@param {type:"string"}
FIRST = ""  #@param {type:"string"}
LAST = ""  #@param {type:"string"}
#@markdown **two_pass のときだけ。B の Video 1 は A の mp4。**
PASS_A = ""  #@param ["", "fast_motion", "combat_motion", "swap_character", "swap_face", "swap_outfit", "real", "orbit360", "two_pass", "text_scene", "join", "affi_template"]
PASS_B = ""  #@param ["", "fast_motion", "combat_motion", "swap_character", "swap_face", "swap_outfit", "real", "orbit360", "two_pass", "text_scene", "join", "affi_template"]
BRANCH = "cursor/h3-sfw-colab-6dc5"  #@param {type:"string"}

import os, shutil, sys, urllib.request
from pathlib import Path

DRIVE_ROOT = "/content/drive/MyDrive/minimax-h3-comfyui"
RAW = f"https://raw.githubusercontent.com/fireworker011/Research/{BRANCH}"

def _plan_path() -> str:
    drive_out = str(Path(DRIVE_ROOT) / "studio" / "latest-plan.json")
    try:
        from google.colab import drive
    except ImportError:
        print("Colab の外。プランは h3_studio_plan.json に書く。")
        return "h3_studio_plan.json"
    drive.mount("/content/drive")
    Path(drive_out).parent.mkdir(parents=True, exist_ok=True)
    return drive_out

def _fetch(rel: str, dest: Path) -> None:
    url = f"{RAW}/{rel}"
    urllib.request.urlretrieve(url, dest)
    if not dest.is_file() or dest.stat().st_size < 50:
        raise SystemExit(f"helper missing: {rel}")

HELPERS = ["colab/h3_sfw.py", "colab/h3_studio.py", "colab/h3_studio_colab_main.py"]
for rel in HELPERS:
    dest = Path("/content") / Path(rel).name
    _fetch(rel, dest)
    print("helper", dest.name)

os.environ["H3_STUDIO_JOB"] = JOB
os.environ["H3_STUDIO_RUNTIME"] = RUNTIME
os.environ["H3_STUDIO_HIGH_MEM"] = "1" if HIGH_MEM else "0"
os.environ["H3_STUDIO_FINISH"] = "1" if FINISH else "0"
os.environ["H3_STUDIO_TURBO"] = "1" if TURBO else "0"
os.environ["H3_STUDIO_DURATION"] = DURATION
os.environ["H3_STUDIO_FAST_SECONDS"] = FAST_SECONDS
os.environ["H3_STUDIO_WITH_ACTION"] = "1" if WITH_ACTION else "0"
os.environ["H3_STUDIO_WITH_COMBAT"] = "1" if WITH_COMBAT else "0"
os.environ["H3_STUDIO_TEMPLATE"] = TEMPLATE
os.environ["H3_STUDIO_PARTS"] = PARTS
os.environ["H3_STUDIO_ACTION"] = ACTION
os.environ["H3_STUDIO_DIALOGUE"] = DIALOGUE
os.environ["H3_STUDIO_HERO_SHEET"] = HERO_SHEET
os.environ["H3_STUDIO_VIDEO"] = VIDEO
os.environ["H3_STUDIO_FIRST"] = FIRST
os.environ["H3_STUDIO_LAST"] = LAST
os.environ["H3_STUDIO_PASS_A"] = PASS_A
os.environ["H3_STUDIO_PASS_B"] = PASS_B
os.environ["H3_STUDIO_GENERATE"] = "0"
os.environ["H3_STUDIO_OUT"] = _plan_path()
for prefix, values in (
    ("HERO", (HERO_HAIR, HERO_COLOR, HERO_RACE, HERO_AGE, HERO_HEIGHT, HERO_WEIGHT, HERO_CLOTHES, HERO_PLACE)),
    ("ENEMY", (ENEMY_HAIR, ENEMY_COLOR, ENEMY_RACE, ENEMY_AGE, ENEMY_HEIGHT, ENEMY_WEIGHT, ENEMY_CLOTHES, ENEMY_PLACE)),
):
    for key, value in zip(("HAIR", "COLOR", "RACE", "AGE", "HEIGHT", "WEIGHT", "CLOTHES", "PLACE"), values):
        os.environ[f"H3_STUDIO_{prefix}_{key}"] = value

sys.path.insert(0, "/content")
from h3_studio_colab_main import main

rc = main()
print("studio exit", rc)
if rc:
    raise SystemExit(rc)


In [ ]:
#@title text_scene を7本書く（生成はしない）
import json
import sys
from pathlib import Path

sys.path.insert(0, "/content")
from h3_studio import GENERATE_MSG, build_text_scenes

plan = build_text_scenes()
if plan.get("generate"):
    raise SystemExit(GENERATE_MSG)
dest = Path("h3_studio_text_scenes.json")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/minimax-h3-comfyui/studio/text-scenes.json")
    dest.parent.mkdir(parents=True, exist_ok=True)
except ImportError:
    print("Colab の外。プランは h3_studio_text_scenes.json に書く。")
dest.write_text(json.dumps(plan, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("WROTE", dest)
print("clips", [clip["scene_id"] for clip in plan["clips"]])
print(GENERATE_MSG)
